# Autoresearch Experiment Analysis

Analysis of autonomous hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, val_bpb, memory_gb, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["val_bpb"] = pd.to_numeric(df["val_bpb"], errors="coerce")
df["memory_gb"] = pd.to_numeric(df["memory_gb"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()
valid = df[df["status"] != "CRASH"].copy().reset_index().rename(columns={"index": "exp_num"})
kept_valid = valid[valid["status"] == "KEEP"].copy()
kept_valid["prev_keep_bpb"] = kept_valid["val_bpb"].shift(1)
reset_points = kept_valid[
    kept_valid["prev_keep_bpb"].notna()
    & (kept_valid["val_bpb"] > kept_valid["prev_keep_bpb"] + 0.02)
]
campaign_start_exp = int(reset_points["exp_num"].iloc[-1]) if not reset_points.empty else int(valid["exp_num"].iloc[0])
campaign = valid[valid["exp_num"] >= campaign_start_exp].copy()
campaign_kept = campaign[campaign["status"] == "KEEP"].copy()

print(f"Total experiments: {len(df)}")
print(f"Campaign starts at experiment #{campaign_start_exp}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    bpb = row["val_bpb"]
    desc = row["description"]
    print(f"  #{i:3d}  bpb={bpb:.6f}  mem={row['memory_gb']:.1f}GB  {desc}")

## Val BPB Over Time

Track how the best (kept) val_bpb evolves as experiments progress. The running minimum shows the "frontier" -- the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Plot discarded experiments from the current campaign as faint background dots.
disc = campaign[campaign["status"] == "DISCARD"]
ax.scatter(disc["exp_num"], disc["val_bpb"],
           c="#cccccc", s=18, alpha=0.7, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots.
ax.scatter(campaign_kept["exp_num"], campaign_kept["val_bpb"],
           c="#2ecc71", s=56, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running minimum step line for the current campaign.
running_min = campaign_kept["val_bpb"].cummin()
ax.step(campaign_kept["exp_num"], running_min, where="post", color="#27ae60",
        linewidth=2, alpha=0.8, zorder=3, label="Running best")

# Label each kept experiment with its description.
for _, row in campaign_kept.iterrows():
    desc = str(row["description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."

    ax.annotate(desc, (row["exp_num"], row["val_bpb"]),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept = len(df[df["status"] == "KEEP"])
y_min = campaign["val_bpb"].min()
y_max = campaign["val_bpb"].max()
margin = max((y_max - y_min) * 0.12, 0.01)

ax.set_xlim(campaign["exp_num"].min() - 0.5, campaign["exp_num"].max() + 0.5)
ax.set_ylim(y_min - margin, y_max + margin)
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Validation BPB (lower is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)
if campaign_start_exp > 0:
    ax.text(0.01, 0.01,
            f"Showing current campaign from experiment #{campaign_start_exp} after baseline reset",
            transform=ax.transAxes, fontsize=9, color="#666666")

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats for the current campaign
baseline_bpb = campaign_kept.iloc[0]["val_bpb"]
best_bpb = campaign_kept["val_bpb"].min()
best_row = campaign_kept.loc[campaign_kept["val_bpb"].idxmin()]

print(f"Baseline val_bpb:  {baseline_bpb:.6f}")
print(f"Best val_bpb:      {best_bpb:.6f}")
print(f"Total improvement: {baseline_bpb - best_bpb:.6f} ({(baseline_bpb - best_bpb) / baseline_bpb * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

# How many experiments to find each improvement
print("Cumulative effort per improvement:")
kept_sorted = campaign_kept.reset_index(drop=True)
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  Experiment #{int(row['exp_num']):3d}: bpb={row['val_bpb']:.6f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta is measured vs the previous kept experiment's bpb
# within the current campaign.
kept = campaign_kept.copy()
kept["prev_bpb"] = kept["val_bpb"].shift(1)
kept["delta"] = kept["prev_bpb"] - kept["val_bpb"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy()

# Sort by delta improvement (biggest first)
hits = hits.sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'BPB':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_bpb']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")